In [1]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for f in sorted(files):
        print(os.path.join(root, f))

/kaggle/input/competitions/workstation-rental-price-tier-prediction/kaggle_ready_dataset/sample_submission.csv
/kaggle/input/competitions/workstation-rental-price-tier-prediction/kaggle_ready_dataset/test.csv
/kaggle/input/competitions/workstation-rental-price-tier-prediction/kaggle_ready_dataset/train.csv


In [2]:
import os
import pandas as pd

train_path = None
test_path = None
sub_path = None

for root, _, files in os.walk("/kaggle/input"):
    for f in files:
        full = os.path.join(root, f)
        if f.startswith("train.csv"):
            train_path = full
        elif f.startswith("test.csv"):
            test_path = full
        elif f.startswith("sample_submission.csv"):
            sub_path = full

print("train:", train_path)
print("test :", test_path)
print("sub  :", sub_path)

assert train_path is not None, "train file not found under /kaggle/input — check the Input panel!"
train_df = pd.read_csv(train_path, low_memory=False)
test_df = pd.read_csv(test_path, low_memory=False)
sub_df = pd.read_csv(sub_path, low_memory=False)
print("Loaded:", train_df.shape, test_df.shape)

train: /kaggle/input/competitions/workstation-rental-price-tier-prediction/kaggle_ready_dataset/train.csv
test : /kaggle/input/competitions/workstation-rental-price-tier-prediction/kaggle_ready_dataset/test.csv
sub  : /kaggle/input/competitions/workstation-rental-price-tier-prediction/kaggle_ready_dataset/sample_submission.csv
Loaded: (279746, 60) (114538, 59)


In [3]:
import os
import time
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
from sklearn.ensemble import HistGradientBoostingClassifier
import lightgbm as lgb

t0 = time.time()

# 2. Lean, high-signal feature engineering (fast vectorized ops only)
def make_fast_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.drop(columns=["price_tier"], errors="ignore").copy()

    if "facilities" in out.columns:
        s = out["facilities"].fillna("").astype(str)
        out["facility_count"] = np.where(s.str.len() > 2, s.str.count(",") + 1, 0).astype("float32")
        out = out.drop(columns=["facilities"])

    for dcol in ["snapshot_start_date", "snapshot_end_date", "first_review_date", "last_review_date"]:
        if dcol in out.columns:
            dt = pd.to_datetime(out[dcol], format="%Y-%m-%d", errors="coerce")
            out[f"{dcol}_yr"] = dt.dt.year.astype("float32")
            out[f"{dcol}_mo"] = dt.dt.month.astype("float32")
            out = out.drop(columns=[dcol])

    if {"max_seats", "workstations"}.issubset(out.columns):
        out["seats_per_ws"] = (out["max_seats"] / (out["workstations"].fillna(0) + 1.0)).astype("float32")
    if {"booked_load", "max_seats"}.issubset(out.columns):
        out["load_per_seat"] = (out["booked_load"] / (out["max_seats"] + 1.0)).astype("float32")
    if {"operator_metro_units", "operator_portfolio_size"}.issubset(out.columns):
        out["metro_share"] = (out["operator_metro_units"] / (out["operator_portfolio_size"].fillna(0) + 1.0)).astype("float32")

    for col, cap in [("max_term_days", 3650.0), ("max_term_avg", 3650.0), ("reviews_per_month", 30.0)]:
        if col in out.columns:
            out[col] = np.clip(out[col].astype("float64"), 0.0, cap).astype("float32")

    return out

y = train_df["price_tier"].astype(np.int8)
X_tr = make_fast_features(train_df)
X_te = make_fast_features(test_df)

# Frequency encoding on key categoricals (no target leakage)
both = pd.concat([X_tr, X_te], axis=0, ignore_index=True)
for col in ["operator_ref", "zone", "space_class", "metro", "borough"]:
    if col in both.columns:
        freq = both[col].value_counts(normalize=True)
        X_tr[f"{col}_freq"] = X_tr[col].map(freq).fillna(0.0).astype("float32")
        X_te[f"{col}_freq"] = X_te[col].map(freq).fillna(0.0).astype("float32")

# Encode remaining object columns as integer codes
for col in X_tr.select_dtypes(include="object").columns:
    vals = pd.concat([X_tr[col], X_te[col]], axis=0).fillna("MISSING").astype(str)
    codes = {v: i for i, v in enumerate(vals.unique())}
    X_tr[col] = X_tr[col].fillna("MISSING").astype(str).map(codes).astype("int32")
    X_te[col] = X_te[col].fillna("MISSING").astype(str).map(codes).astype("int32")

print(f"Features ready in {time.time() - t0:.1f}s | cols={X_tr.shape[1]}")

# 3. 3-Fold Stratified CV with HGB + LightGBM
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
oof_hgb = np.zeros((len(X_tr), 4), dtype=np.float64)
oof_lgb = np.zeros((len(X_tr), 4), dtype=np.float64)
test_hgb = np.zeros((len(X_te), 4), dtype=np.float64)
test_lgb = np.zeros((len(X_te), 4), dtype=np.float64)

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_tr, y)):
    t_f = time.time()
    hgb = HistGradientBoostingClassifier(
        learning_rate=0.08, max_iter=200, max_leaf_nodes=63,
        min_samples_leaf=30, l2_regularization=1.0, random_state=42 + fold
    )
    hgb.fit(X_tr.iloc[tr_idx], y.iloc[tr_idx])
    oof_hgb[va_idx] = hgb.predict_proba(X_tr.iloc[va_idx])
    test_hgb += hgb.predict_proba(X_te) / 3.0

    gbm = lgb.LGBMClassifier(
        n_estimators=250, learning_rate=0.07, num_leaves=63,
        min_child_samples=30, subsample=0.8, subsample_freq=1,
        colsample_bytree=0.8, n_jobs=-1, random_state=42 + fold, verbose=-1
    )
    gbm.fit(X_tr.iloc[tr_idx], y.iloc[tr_idx] - 1)
    oof_lgb[va_idx] = gbm.predict_proba(X_tr.iloc[va_idx])
    test_lgb += gbm.predict_proba(X_te) / 3.0

    print(f"Fold {fold + 1} done in {time.time() - t_f:.1f}s")

print(f"3-Fold OOF Acc - HGB: {accuracy_score(y, oof_hgb.argmax(axis=1) + 1):.5f}")
print(f"3-Fold OOF Acc - LGB: {accuracy_score(y, oof_lgb.argmax(axis=1) + 1):.5f}")

blend_proba = 0.5 * test_hgb + 0.5 * test_lgb
sub = pd.DataFrame({
    "ID": sub_df["ID"],
    "price_tier": (blend_proba.argmax(axis=1) + 1).astype(int),
})
sub.to_csv("submission.csv", index=False)
print(f"Total runtime: {time.time() - t0:.1f}s | Saved submission.csv ({sub.shape})")

Features ready in 4.1s | cols=71
Fold 1 done in 104.4s
Fold 2 done in 110.8s
Fold 3 done in 102.1s
3-Fold OOF Acc - HGB: 0.71533
3-Fold OOF Acc - LGB: 0.71633
Total runtime: 321.6s | Saved submission.csv ((114538, 2))
